# 🚀 Missão Aurora Siger — Relatório Operacional de Pré-Decolagem

**Curso:** Ciência da Computação — FIAP  
**Participantes:** Julia Ramos, Carlos Eugenio, Julio Guma, Matheus Fuchens  
**Fase:** 1 — Atividade Integradora

---

Relatório de pré-decolagem da missão fictícia Aurora Siger. Seções:

1. Organização e descrição da telemetria
2. Algoritmo de verificação (ver `README.md`)
3. Script Python de verificação
4. Análise energética
5. Análise assistida por IA (scikit-learn)
6. Reflexão crítica (ver `README.md`)

> Valores marcados como `# SIMULATED` foram estimados por ordem de grandeza. Ver `telemetry.md`.

In [ ]:
# =============================================================================
# IMPORTS — all dependencies declared upfront to avoid cell-order issues
# =============================================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

# =============================================================================
# GLOBAL CONSTANTS
# =============================================================================

RANDOM_STATE: int = 42   # fixed seed — makes all models and splits reproducible
REPORT_WIDTH: int = 60   # character width used by every section separator

SEPARATOR: str = "=" * REPORT_WIDTH

np.random.seed(RANDOM_STATE)

# =============================================================================
# DISPLAY HELPERS
# =============================================================================


def format_status(flag: bool) -> str:
    """Return a human-readable status string for a boolean flag."""
    return "OK" if flag else "FAIL"


In [ ]:
# =============================================================================
# DEPENDENCY CHECK — run this cell before anything else
# =============================================================================

import importlib

REQUIRED_PACKAGES = ["numpy", "pandas", "sklearn"]

missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if missing:
    install_cmd = "pip install " + " ".join(
        pkg.replace("sklearn", "scikit-learn") for pkg in missing
    )
    print(f"Missing packages : {missing}")
    print(f"Run to install   : {install_cmd}")
else:
    print(f"All dependencies found: {REQUIRED_PACKAGES}")
    print("Ready to execute.")

---
## Seção 1 — Organização e Descrição da Telemetria

Dados interpretados:
- Temperatura interna e externa (°C)
- Integridade estrutural (flag binário 0/1)
- Nível de energia (%)
- Pressão dos tanques (bar)
- Status dos módulos críticos (booleano por subsistema)

In [ ]:
# =============================================================================
# SECTION 1 — TELEMETRY DATA
# Sources documented in telemetry.md
# =============================================================================


@dataclass(frozen=True)
class TelemetryReading:
    """Immutable snapshot of spacecraft telemetry at pre-launch verification."""

    # Temperatures (°C) — Source: MIT OCW Satellite Engineering (2003)
    internal_temperature: float   # Electronics bay
    external_temperature: float   # Hull / structural skin

    # Structural integrity flag — Source: ESA Mars Express right_flag model
    structural_integrity: int     # 1 = nominal, 0 = failure

    # Energy level (%) — Source: ESA Advanced Concepts Team (2021)
    energy_level_pct: float

    # Propulsion tank pressure (bar) — Source: NASA SBIR (SIMULATED order of magnitude)
    tank_pressure_bar: float

    # Critical subsystem flags — Source: ESA Mars Express ESOC subsystems
    propulsion_ok: bool
    power_ok: bool
    comms_ok: bool
    thermal_ok: bool
    navigation_ok: bool


# --- Simulated telemetry reading at T-0 ---
TELEMETRY = TelemetryReading(
    internal_temperature=22.5,    # °C — nominal, within -20 to +70°C
    external_temperature=-45.0,   # °C — nominal, within -100 to +125°C
    structural_integrity=1,       # 1 = intact
    energy_level_pct=87.3,        # % — above 60% minimum
    tank_pressure_bar=34.8,       # bar — within 20–40 bar
    propulsion_ok=True,
    power_ok=True,
    comms_ok=True,
    thermal_ok=True,
    navigation_ok=True,
)

# --- Energy system parameters (SIMULATED — order of magnitude) ---
TOTAL_CAPACITY_KWH: float    = 120.0  # kWh — medium-class spacecraft reference
LAUNCH_CONSUMPTION_KW: float =   8.5  # kW  — estimated peak draw at launch
SYSTEM_EFFICIENCY: float     =  0.92  # η   — Cap. 7 FIAP; new systems ~92%
LOSS_FACTOR: float           =  0.08  # 8% resistive losses — Cap. 7 FIAP: P = I²×R


def print_telemetry(reading: TelemetryReading) -> None:
    """Print a formatted telemetry snapshot. Display-only, no return value."""
    print(SEPARATOR)
    print("AURORA SIGER — PRE-LAUNCH TELEMETRY SNAPSHOT")
    print(SEPARATOR)
    print(f"  Internal temperature : {reading.internal_temperature:>8.1f} °C")
    print(f"  External temperature : {reading.external_temperature:>8.1f} °C")
    print(f"  Structural integrity : {reading.structural_integrity:>8}   (1=OK, 0=FAIL)")
    print(f"  Energy level         : {reading.energy_level_pct:>8.1f} %")
    print(f"  Tank pressure        : {reading.tank_pressure_bar:>8.1f} bar")
    print(f"  Propulsion system    : {format_status(reading.propulsion_ok):>8}")
    print(f"  Power management     : {format_status(reading.power_ok):>8}")
    print(f"  Communications       : {format_status(reading.comms_ok):>8}")
    print(f"  Thermal control      : {format_status(reading.thermal_ok):>8}")
    print(f"  Navigation / ADCS    : {format_status(reading.navigation_ok):>8}")
    print(SEPARATOR)


print_telemetry(TELEMETRY)

---
## Seção 2 — Algoritmo de Verificação

Fluxograma completo em `README.md`. Cada parâmetro tem uma função pura que retorna `bool`; `verify_launch_readiness()` consolida todas.

```
check_internal_temperature(value)  → bool
check_external_temperature(value)  → bool
check_structural_integrity(flag)   → bool
check_energy_level(pct)            → bool
check_tank_pressure(bar)           → bool
check_critical_modules(...)        → bool
check_autonomy(hours)              → bool
verify_launch_readiness(...)       → Tuple[str, Dict[str, bool]]
```

---
## Seção 3 — Script Python de Verificação

In [ ]:
# =============================================================================
# SECTION 3 — VERIFICATION SCRIPT (Functional Programming)
# =============================================================================
# Safety thresholds are the single source of truth for verification logic
# AND for the AI dataset generation in Section 5.
# Sources documented in telemetry.md
# =============================================================================

# --- Safety thresholds ---
# Source: MIT OCW Satellite Engineering (2003); ESA Bulletin 87
TEMP_INTERNAL_MIN: float = -20.0
TEMP_INTERNAL_MAX: float =  70.0
TEMP_EXTERNAL_MIN: float = -100.0
TEMP_EXTERNAL_MAX: float =  125.0

# Source: ESA Advanced Concepts Team (2021)
ENERGY_MIN_PCT: float = 60.0

# Source: NASA SBIR Thermal Management (SIMULATED order of magnitude)
PRESSURE_MIN_BAR: float = 20.0
PRESSURE_MAX_BAR: float = 40.0

# Minimum autonomy required at launch
AUTONOMY_MIN_HOURS: float = 2.0

# Launch decision strings — exact strings required by activity specification
DECISION_READY:   str = "PRONTO PARA DECOLAR"
DECISION_ABORTED: str = "DECOLAGEM ABORTADA"


# --- Pure check functions — each returns bool, no side effects ---

def check_internal_temperature(value: float) -> bool:
    """Return True if internal electronics temperature is within operational range.

    Range: -20°C to +70°C
    Source: MIT OCW Satellite Engineering (2003) — digital electronics
    """
    return TEMP_INTERNAL_MIN <= value <= TEMP_INTERNAL_MAX


def check_external_temperature(value: float) -> bool:
    """Return True if external hull temperature is within survival range.

    Range: -100°C to +125°C
    Source: MIT OCW / ESA Bulletin 87 — structural panels
    """
    return TEMP_EXTERNAL_MIN <= value <= TEMP_EXTERNAL_MAX


def check_structural_integrity(flag: int) -> bool:
    """Return True if structural integrity flag indicates nominal state.

    Expected: 1 (nominal). 0 indicates detected structural failure.
    Source: ESA Mars Express dataset — binary status flag model
    """
    return flag == 1


def check_energy_level(pct: float) -> bool:
    """Return True if battery charge meets minimum launch threshold.

    Minimum: 60%
    Source: ESA Advanced Concepts Team (2021) — operational safety margin
    """
    return pct >= ENERGY_MIN_PCT


def check_tank_pressure(bar: float) -> bool:
    """Return True if propulsion tank pressure is within operational range.

    Range: 20 to 40 bar
    Source: NASA SBIR Thermal Management (SIMULATED)
    """
    return PRESSURE_MIN_BAR <= bar <= PRESSURE_MAX_BAR


def check_critical_modules(reading: TelemetryReading) -> bool:
    """Return True only if ALL critical subsystems are operational.

    Subsystems: propulsion, power management, communications,
                thermal control, navigation/ADCS.
    Source: ESA Mars Express ESOC — primary subsystem list
    """
    return all([
        reading.propulsion_ok,
        reading.power_ok,
        reading.comms_ok,
        reading.thermal_ok,
        reading.navigation_ok,
    ])


def check_autonomy(autonomy_hours: float) -> bool:
    """Return True if calculated energy autonomy meets minimum mission requirement.

    Minimum: 2.0 hours
    """
    return autonomy_hours >= AUTONOMY_MIN_HOURS


def run_checks(
    reading: TelemetryReading,
    autonomy_hours: float,
) -> Dict[str, bool]:
    """Return individual pass/fail result for each pre-launch check.

    Pure data — no formatting, no decision string.
    """
    return {
        "internal_temperature": check_internal_temperature(reading.internal_temperature),
        "external_temperature": check_external_temperature(reading.external_temperature),
        "structural_integrity": check_structural_integrity(reading.structural_integrity),
        "energy_level"        : check_energy_level(reading.energy_level_pct),
        "tank_pressure"       : check_tank_pressure(reading.tank_pressure_bar),
        "critical_modules"    : check_critical_modules(reading),
        "autonomy"            : check_autonomy(autonomy_hours),
    }


def decide_launch(checks: Dict[str, bool]) -> str:
    """Return DECISION_READY if all checks passed, DECISION_ABORTED otherwise."""
    return DECISION_READY if all(checks.values()) else DECISION_ABORTED


def verify_launch_readiness(
    reading: TelemetryReading,
    autonomy_hours: float,
) -> Tuple[str, Dict[str, bool]]:
    """Run all pre-launch checks and return final launch decision.

    Returns:
        Tuple of (decision_string, checks_dict)
        decision_string: DECISION_READY | DECISION_ABORTED
        checks_dict: individual pass/fail result per check
    """
    checks = run_checks(reading, autonomy_hours)
    return decide_launch(checks), checks


def print_checklist(decision: str, checks: Dict[str, bool]) -> None:
    """Print formatted verification checklist. Display-only, no return value."""
    print(SEPARATOR)
    print("PRE-LAUNCH VERIFICATION CHECKLIST")
    print(SEPARATOR)
    for name, passed in checks.items():
        icon   = "✅" if passed else "❌"
        result = "PASS" if passed else "FAIL"
        print(f"  {icon}  {name.replace('_', ' ').title():<30} {result}")
    print(SEPARATOR)
    print(f"\n  >>> {decision} <<<\n")
    print(SEPARATOR)

---
## Seção 4 — Análise Energética

In [ ]:
# =============================================================================
# SECTION 4 — ENERGY ANALYSIS
# =============================================================================
# Formulas from Cap. 7 FIAP — Fundamentos da Eficiência Energética:
#   η  = E_util / E_total                    (system efficiency)
#   P  = I² × R                              (resistive thermal losses)
#   FC = demanda_media / demanda_maxima      (load factor)
# =============================================================================


@dataclass(frozen=True)
class EnergyAnalysis:
    """Immutable result of the pre-launch energy calculation."""
    total_capacity_kwh:   float
    charge_pct:           float
    current_charge_kwh:   float
    energy_losses_kwh:    float
    efficiency:           float
    energy_available_kwh: float
    consumption_kw:       float
    autonomy_hours:       float
    load_factor:          float


def compute_energy_analysis(
    total_capacity_kwh: float,
    charge_pct: float,
    consumption_kw: float,
    efficiency: float,
    loss_factor: float,
) -> EnergyAnalysis:
    """Compute full energy analysis for pre-launch verification.

    Returns EnergyAnalysis with all derived metrics (Cap. 7 FIAP formulas).
    """
    # Cap. 7 FIAP — carga_atual = capacidade × (carga_pct / 100)
    current_charge_kwh = total_capacity_kwh * (charge_pct / 100)

    # Cap. 7 FIAP — P = I²×R (resistive thermal dissipation)
    energy_losses_kwh = current_charge_kwh * loss_factor

    # Cap. 7 FIAP — η = E_util / E_total
    energy_available_kwh = current_charge_kwh * efficiency

    # Autonomy = available energy / launch power draw
    autonomy_hours = energy_available_kwh / consumption_kw

    # Cap. 7 FIAP — FC = demanda_media / demanda_maxima
    load_factor = current_charge_kwh / total_capacity_kwh

    return EnergyAnalysis(
        total_capacity_kwh=total_capacity_kwh,
        charge_pct=charge_pct,
        current_charge_kwh=current_charge_kwh,
        energy_losses_kwh=energy_losses_kwh,
        efficiency=efficiency,
        energy_available_kwh=energy_available_kwh,
        consumption_kw=consumption_kw,
        autonomy_hours=autonomy_hours,
        load_factor=load_factor,
    )


def print_energy_analysis(ea: EnergyAnalysis) -> None:
    """Print formatted energy analysis report. Display-only, no return value."""
    autonomy_ok = ea.autonomy_hours >= AUTONOMY_MIN_HOURS
    print(SEPARATOR)
    print("ENERGY ANALYSIS — AURORA SIGER")
    print(SEPARATOR)
    print(f"  Total capacity       : {ea.total_capacity_kwh:>8.1f} kWh")
    print(f"  Current charge       : {ea.charge_pct:>8.1f} %")
    print(f"  Current charge       : {ea.current_charge_kwh:>8.2f} kWh")
    print(f"  Energy losses (I²R)  : {ea.energy_losses_kwh:>8.2f} kWh  ({LOSS_FACTOR:.0%} loss factor)")
    print(f"  System efficiency η  : {ea.efficiency:>8.0%}")
    print(f"  Energy available     : {ea.energy_available_kwh:>8.2f} kWh")
    print(f"  Launch consumption   : {ea.consumption_kw:>8.1f} kW")
    print(f"  Autonomy             : {ea.autonomy_hours:>8.2f} hours")
    print(f"  Load factor (FC)     : {ea.load_factor:>8.2f}")
    print(SEPARATOR)
    print(f"  Minimum required     : {AUTONOMY_MIN_HOURS:>8.1f} hours")
    print(f"  Status               : {format_status(autonomy_ok)}")
    print(SEPARATOR)


# --- Compute energy, then run full verification ---
energy = compute_energy_analysis(
    total_capacity_kwh=TOTAL_CAPACITY_KWH,
    charge_pct=TELEMETRY.energy_level_pct,
    consumption_kw=LAUNCH_CONSUMPTION_KW,
    efficiency=SYSTEM_EFFICIENCY,
    loss_factor=LOSS_FACTOR,
)

decision, checks = verify_launch_readiness(TELEMETRY, energy.autonomy_hours)

print_energy_analysis(energy)
print()
print_checklist(decision, checks)

---
## Seção 5 — Análise Assistida por IA

Dois algoritmos complementares:

1. **IsolationForest** (não supervisionado) — detecta desvios do padrão nominal sem precisar de rótulos históricos.
2. **DecisionTreeClassifier** (supervisionado) — aprende regras if-then a partir de histórico rotulado; geometricamente compatível com o padrão de anomalias (OR de condições por feature).

Dataset sintético gerado a partir das constantes da Seção 3 — consistência garantida entre verificação e treinamento. **Recall é a métrica prioritária**: falso negativo é mais perigoso que falso positivo.

In [ ]:
# =============================================================================
# SECTION 5 — AI-ASSISTED ANALYSIS
# =============================================================================
# Dataset: synthetic, generated from the safety thresholds defined in Section 3
# Sources: NASA SMAP/MSL (Hundman et al., KDD 2018); MIT OCW; ESA MEX
# =============================================================================

# Feature column order — shared by both dataset generators and classifiers
FEATURE_NAMES: List[str] = [
    "internal_temp",
    "external_temp",
    "structural_integrity",
    "energy_pct",
    "tank_pressure",
]

# Nominal sampling ranges — kept slightly inside safe limits to represent typical ops.
# Derived from Section 3 threshold constants: single source of truth.
NOMINAL_RANGES: Dict[str, Tuple[float, float]] = {
    "internal_temp"       : (TEMP_INTERNAL_MIN + 5,  TEMP_INTERNAL_MAX - 5),
    "external_temp"       : (TEMP_EXTERNAL_MIN + 10, TEMP_EXTERNAL_MAX - 15),
    "structural_integrity": (1.0, 1.0),
    "energy_pct"          : (ENERGY_MIN_PCT + 2,     100.0),
    "tank_pressure"       : (PRESSURE_MIN_BAR + 2,   PRESSURE_MAX_BAR - 2),
}

# Anomaly injection ranges — deliberately outside safe limits.
# Each feature maps to one or more (lo, hi) out-of-range bands.
# Inspired by anomaly classes in NASA SMAP/MSL dataset (Hundman et al., KDD 2018).
ANOMALY_RANGES: Dict[str, List[Tuple[float, float]]] = {
    "internal_temp"       : [(TEMP_INTERNAL_MIN - 30, TEMP_INTERNAL_MIN - 1),
                             (TEMP_INTERNAL_MAX + 1,  TEMP_INTERNAL_MAX + 40)],
    "external_temp"       : [(TEMP_EXTERNAL_MIN - 80, TEMP_EXTERNAL_MIN - 1),
                             (TEMP_EXTERNAL_MAX + 1,  TEMP_EXTERNAL_MAX + 75)],
    "structural_integrity": [(0.0, 0.0)],
    "energy_pct"          : [(5.0, ENERGY_MIN_PCT - 1)],
    "tank_pressure"       : [(0.0,                 PRESSURE_MIN_BAR - 1),
                             (PRESSURE_MAX_BAR + 1, PRESSURE_MAX_BAR + 40)],
}


# --- Pure dataset generation functions ---

def _sample_uniform(lo: float, hi: float) -> float:
    """Sample a single float uniformly from [lo, hi]."""
    return float(np.random.uniform(lo, hi))


def _sample_nominal(feature: str) -> float:
    """Return one nominal (safe) value for the given feature."""
    lo, hi = NOMINAL_RANGES[feature]
    return _sample_uniform(lo, hi)


def _sample_anomaly(feature: str) -> float:
    """Return one anomalous value for the given feature.

    Picks a random out-of-range band, then samples from it.
    """
    bands   = ANOMALY_RANGES[feature]
    lo, hi  = bands[np.random.randint(len(bands))]
    return _sample_uniform(lo, hi)


def _inject_fault(row: np.ndarray, fault_feature: str) -> np.ndarray:
    """Return a copy of row with one feature replaced by an anomalous value."""
    col    = FEATURE_NAMES.index(fault_feature)
    result = row.copy()
    result[col] = _sample_anomaly(fault_feature)
    return result


def telemetry_to_feature_array(reading: TelemetryReading) -> np.ndarray:
    """Convert a TelemetryReading into the feature array expected by the models."""
    return np.array([[
        reading.internal_temperature,
        reading.external_temperature,
        reading.structural_integrity,
        reading.energy_level_pct,
        reading.tank_pressure_bar,
    ]])


def generate_nominal_missions(n: int) -> np.ndarray:
    """Generate n rows of nominal telemetry within safe operational ranges."""
    return np.array(
        [[_sample_nominal(f) for f in FEATURE_NAMES] for _ in range(n)]
    )


def generate_anomalous_missions(n: int) -> np.ndarray:
    """Generate n rows of telemetry each with exactly one injected fault.

    Fault feature is chosen uniformly at random per row.
    Source: NASA SMAP/MSL anomaly structure — Hundman et al., KDD 2018.
    """
    fault_features = list(ANOMALY_RANGES.keys())
    chosen_faults  = np.random.choice(fault_features, size=n)
    nominal_rows   = generate_nominal_missions(n)

    return np.array([
        _inject_fault(row, fault)
        for row, fault in zip(nominal_rows, chosen_faults)
    ])


# Dataset label constants
LABEL_NOMINAL: int = 0   # mission within all safe operational limits
LABEL_ANOMALY: int = 1   # mission with at least one parameter out of range


def build_dataset(
    n_nominal: int,
    n_anomalous: int,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Build labeled feature matrix X, label vector y, and summary DataFrame."""
    X_nominal   = generate_nominal_missions(n_nominal)
    X_anomalous = generate_anomalous_missions(n_anomalous)

    X = np.vstack([X_nominal, X_anomalous])
    y = np.array([LABEL_NOMINAL] * n_nominal + [LABEL_ANOMALY] * n_anomalous)

    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df["label"]      = y
    df["label_name"] = df["label"].map({LABEL_NOMINAL: "nominal", LABEL_ANOMALY: "anomaly"})

    return X, y, df


# --- Build dataset ---
# 2000 missions: 1500 nominal (75%) + 500 anomalous (25%).
# With 5-fold CV: each test fold has ~100 anomalies, keeping recall std below 3%.
N_NOMINAL:   int = 1500
N_ANOMALOUS: int = 500

X, y, df = build_dataset(N_NOMINAL, N_ANOMALOUS)

print(f"Dataset: {N_NOMINAL} nominal + {N_ANOMALOUS} anomalous = {len(df)} missions")
print(df.groupby("label_name")[FEATURE_NAMES].mean().round(2))

In [ ]:
# =============================================================================
# 5A — ANOMALY DETECTION: IsolationForest (unsupervised)
# =============================================================================

# IsolationForest output convention — documented here so the remapping is clear
IF_ANOMALY_LABEL: int = -1   # IsolationForest marks anomalies as -1
IF_INLIER_LABEL:  int = +1   # IsolationForest marks nominal as +1

CONTAMINATION_RATIO: float = N_ANOMALOUS / (N_NOMINAL + N_ANOMALOUS)


@dataclass(frozen=True)
class IsolationForestResult:
    """Immutable result of IsolationForest training and classification."""
    scaler:    StandardScaler
    model:     IsolationForest
    labels:    np.ndarray     # remapped: LABEL_NOMINAL=0, LABEL_ANOMALY=1
    recall:    float
    precision: float
    f1:        float
    cm:        np.ndarray


def train_isolation_forest(
    X: np.ndarray,
    y: np.ndarray,
    contamination: float,
) -> IsolationForestResult:
    """Fit an IsolationForest and evaluate against ground-truth labels.

    Returns an immutable result object — no global state modified.
    """
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model  = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
    preds  = model.fit_predict(X_scaled)

    # Remap IsolationForest output to the project label convention
    labels = (preds == IF_ANOMALY_LABEL).astype(int)

    return IsolationForestResult(
        scaler=scaler,
        model=model,
        labels=labels,
        recall=recall_score(y, labels),
        precision=precision_score(y, labels),
        f1=f1_score(y, labels),
        cm=confusion_matrix(y, labels),
    )


def classify_with_isolation_forest(
    result: IsolationForestResult,
    reading: TelemetryReading,
) -> str:
    """Classify a single telemetry reading. Returns 'ANOMALY DETECTED' or 'NOMINAL'."""
    features = telemetry_to_feature_array(reading)
    scaled   = result.scaler.transform(features)
    pred     = result.model.predict(scaled)[0]
    return "ANOMALY DETECTED" if pred == IF_ANOMALY_LABEL else "NOMINAL"


def print_isolation_forest_report(
    result: IsolationForestResult,
    mission_classification: str,
) -> None:
    """Print IsolationForest evaluation report. Display-only."""
    print(SEPARATOR)
    print("IsolationForest — Anomaly Detection (unsupervised)")
    print(SEPARATOR)
    print(f"  Recall    : {result.recall:.2%}   ← priority metric for safety-critical systems")
    print(f"  Precision : {result.precision:.2%}")
    print(f"  F1 Score  : {result.f1:.2%}")
    print()
    print("  Confusion Matrix (rows=actual, cols=predicted):")
    print("                   Nominal  Anomaly")
    print(f"  Actual Nominal : {result.cm[0][0]:>6}   {result.cm[0][1]:>6}")
    print(f"  Actual Anomaly : {result.cm[1][0]:>6}   {result.cm[1][1]:>6}")
    print(SEPARATOR)
    print(f"  Current mission: {mission_classification}")


iso        = train_isolation_forest(X, y, CONTAMINATION_RATIO)
iso_result = classify_with_isolation_forest(iso, TELEMETRY)
print_isolation_forest_report(iso, iso_result)


In [ ]:
# =============================================================================
# 5B — BINARY CLASSIFICATION: DecisionTreeClassifier (supervised)
# =============================================================================
# Decision trees learn if-then rules — same structure as the Section 3 checklist.
# Trained here, the tree rediscovers safety thresholds autonomously.
# Scale-invariant: no feature scaling needed.
#
# Design: class_weight='balanced' (3:1 imbalance), StratifiedKFold CV (no leakage).
# =============================================================================

TEST_SIZE:    float = 0.25   # fraction of data held out for hold-out evaluation
CV_FOLDS:     int   = 5      # number of stratified folds for cross-validation
DT_MAX_DEPTH: int   = 10     # upper bound on tree depth; actual depth may be lower
TREE_DISPLAY_DEPTH: int = 3  # levels shown in the printed rule summary


@dataclass(frozen=True)
class DecisionTreeResult:
    """Immutable result of DecisionTreeClassifier training and classification."""
    model:          DecisionTreeClassifier
    y_test:         np.ndarray
    y_pred:         np.ndarray
    recall:         float
    precision:      float
    f1:             float
    mission_label:  int
    mission_proba:  np.ndarray
    cv_recall_mean: float   # mean recall across CV_FOLDS stratified folds
    cv_recall_std:  float   # standard deviation — lower means more stable estimate
    tree_depth:     int     # actual depth learned by the model
    n_leaves:       int     # number of leaf nodes (model complexity indicator)


def train_decision_tree(
    X: np.ndarray,
    y: np.ndarray,
    reading: TelemetryReading,
    test_size: float,
    cv_folds: int,
) -> DecisionTreeResult:
    """Train a DecisionTreeClassifier and classify a single telemetry reading.

    Returns an immutable result object — no global state modified.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    model = DecisionTreeClassifier(
        max_depth=DT_MAX_DEPTH,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Cross-validation — no Pipeline needed: DecisionTree is scale-invariant
    kfold     = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(
        DecisionTreeClassifier(
            max_depth=DT_MAX_DEPTH,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        X, y, cv=kfold, scoring="recall",
    )

    features      = telemetry_to_feature_array(reading)
    mission_label = model.predict(features)[0]
    mission_proba = model.predict_proba(features)[0]

    return DecisionTreeResult(
        model=model,
        y_test=y_test,
        y_pred=y_pred,
        recall=recall_score(y_test, y_pred),
        precision=precision_score(y_test, y_pred),
        f1=f1_score(y_test, y_pred),
        mission_label=int(mission_label),
        mission_proba=mission_proba,
        cv_recall_mean=float(cv_scores.mean()),
        cv_recall_std=float(cv_scores.std()),
        tree_depth=int(model.get_depth()),
        n_leaves=int(model.get_n_leaves()),
    )


def print_decision_tree_report(
    result: DecisionTreeResult,
    feature_names: List[str],
) -> None:
    """Print DecisionTree evaluation report including learned rules. Display-only."""
    dt_classification = "ANOMALY DETECTED" if result.mission_label == LABEL_ANOMALY else "NOMINAL"

    print(SEPARATOR)
    print("DecisionTreeClassifier — Binary Classification (supervised)")
    print(SEPARATOR)
    print(classification_report(
        result.y_test, result.y_pred, target_names=["nominal", "anomaly"]
    ))
    print(f"  Hold-out recall (anomaly)   : {result.recall:.2%}   ← single split")
    print(f"  CV recall {CV_FOLDS}-fold mean       : {result.cv_recall_mean:.2%} ± {result.cv_recall_std:.2%}   ← more robust estimate")
    print(f"  Precision (anomaly)         : {result.precision:.2%}")
    print(f"  F1 Score  (anomaly)         : {result.f1:.2%}")
    print(f"  Tree depth (actual)         : {result.tree_depth}")
    print(f"  Leaf nodes                  : {result.n_leaves}")
    print(SEPARATOR)
    print(f"  Current mission     : {dt_classification}")
    print(f"  Probability nominal : {result.mission_proba[0]:.2%}")
    print(f"  Probability anomaly : {result.mission_proba[1]:.2%}")
    print()
    print(f"  Learned decision rules (first {TREE_DISPLAY_DEPTH} levels):")
    print(export_text(result.model, feature_names=feature_names, max_depth=TREE_DISPLAY_DEPTH))


dt_result = train_decision_tree(X, y, TELEMETRY, TEST_SIZE, CV_FOLDS)
print_decision_tree_report(dt_result, FEATURE_NAMES)


In [ ]:
# =============================================================================
# 5C — AI RISK SUMMARY
# Covers: data classification, anomaly identification, risk suggestion
# =============================================================================

# Risk level thresholds
RISK_LOW_THRESHOLD:      float = 0.20  # below 20% anomaly probability → LOW
RISK_MODERATE_THRESHOLD: float = 0.50  # below 50% → MODERATE; above → HIGH


def classify_risk_level(anomaly_probability: float) -> str:
    """Return risk level string based on anomaly probability."""
    if anomaly_probability < RISK_LOW_THRESHOLD:
        return "LOW"
    if anomaly_probability < RISK_MODERATE_THRESHOLD:
        return "MODERATE"
    return "HIGH"


def build_ai_risk_summary(
    isolation_forest_result: str,
    isolation_forest_recall: float,
    dt_result: DecisionTreeResult,
) -> str:
    """Build and return the AI risk assessment string.

    Pure function — no side effects.
    """
    dt_classification = "ANOMALY DETECTED" if dt_result.mission_label == LABEL_ANOMALY else "NOMINAL"
    anomaly_prob      = dt_result.mission_proba[1]
    risk_level        = classify_risk_level(anomaly_prob)
    consensus         = isolation_forest_result == dt_classification
    recommendation    = "PROCEED WITH CAUTION" if risk_level == "LOW" else "HOLD AND REVIEW"

    lines = [
        SEPARATOR,
        "AI RISK ASSESSMENT — AURORA SIGER",
        SEPARATOR,
        "  Data classification",
        f"    IsolationForest (unsupervised) : {isolation_forest_result}",
        f"    DecisionTree    (supervised)   : {dt_classification}",
        f"    Model consensus               : {'YES' if consensus else 'NO — REVIEW REQUIRED'}",
        "",
        "  Anomaly identification",
        f"    Anomaly probability           : {anomaly_prob:.1%}",
        f"    IsolationForest recall        : {isolation_forest_recall:.1%}",
        f"    DecisionTree recall           : {dt_result.recall:.1%}",
        "",
        "  Risk suggestion",
        f"    Risk level                    : {risk_level}",
        f"    Recommended action            : {recommendation}",
        "",
        SEPARATOR,
        "  NOTE: AI analysis is advisory only. Final launch decision",
        "  must be made by mission operators. Automated systems must",
        "  not override human judgment in safety-critical contexts.",
        "  (ISO 26000 — accountability; Cap. 6 FIAP — interpretability)",
        SEPARATOR,
    ]
    return "\n".join(lines)


print(build_ai_risk_summary(iso_result, iso.recall, dt_result))


---
## Seção 6 — Reflexão Crítica

> Reflexão completa em [`README.md`](README.md) — seção **Reflexão Crítica**.

In [ ]:
# =============================================================================
# FINAL OUTPUT — COMPLETE MISSION REPORT SUMMARY
# =============================================================================

def print_final_report(
    ea: EnergyAnalysis,
    iso_result: str,
    dt_result: DecisionTreeResult,
    decision: str,
) -> None:
    """Print the complete mission pre-launch report summary. Display-only."""
    dt_classification = "ANOMALY DETECTED" if dt_result.mission_label == LABEL_ANOMALY else "NOMINAL"
    border = "*" * REPORT_WIDTH

    print("\n" + SEPARATOR)
    print("  AURORA SIGER — FINAL PRE-LAUNCH REPORT")
    print(SEPARATOR)
    print(f"  Energy available     : {ea.energy_available_kwh:.2f} kWh")
    print(f"  Autonomy             : {ea.autonomy_hours:.2f} h")
    print(f"  IsolationForest      : {iso_result}")
    print(f"  DecisionTree         : {dt_classification}")
    print(f"  Anomaly probability  : {dt_result.mission_proba[1]:.1%}")
    print()
    print(border)
    print(f"  >>> {decision} <<<")
    print(border)


print_final_report(energy, iso_result, dt_result, decision)
